Author: ZheKai Version: 0.1

This version only deals with two dataset: crc01, crc03
Further Data will be integrated once the whole pipeline is built

In [21]:
### Import modules
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import anndata as ad
import sys
import re
### Import personal Utensils
sys.path.insert(0, "..")
import utils as ut

### Set Data Dir

In [22]:
DataRootDir = Path("../../data")

### Data Curation
Now integrate each individual data \

Unification of metadata
- Glob_CellId (used as index): study name + barcode
- Glob_Sample: study name + sample
- Glob_Patient: study name + patient
- Patient: the name of the patient, use the name in the original study
- Study: study name in ./raw-data
- Tissue: crc or pm

It is decided that in Anndata, .X stores log transformed data, while .layers\['counts'] stores raw counts

Besides, we have to unify gene symbols. Since projecting ENSEMBL ID to gene symbol loses 

---
!!! for crc01, patient marked with 'l' means that liver metastasis is also collected

#### Step one: read in and unify data type and meta data(obs)

In [23]:
# Read-in files
CurrentDir = DataRootDir / "crc01"
adata_crc01 = sc.read_h5ad(CurrentDir / "HRA003293.h5ad")

### crc01 data
adata_crc01.X = ut.to_float32(adata_crc01.layers["data"])
del adata_crc01.layers["data"]
adata_crc01.layers["counts"] = ut.to_int32_counts(
    adata_crc01.layers["counts"]
)

# modify obs
crc01_obs = adata_crc01.obs.copy()
obs_list = ['orig.ident','sample','cells','tissue','patient','tissuegroup','tumorgroup','site']
crc01_obs = crc01_obs[obs_list]
crc01_obs['Study'] = 'crc01'
crc01_obs = crc01_obs.rename(columns={"patient": "Patient","tissue":"Tissue","sample":"Sample"})
crc01_obs['Glob_CellID'] = crc01_obs['Study'] + '-' + crc01_obs['cells']
crc01_obs = crc01_obs.drop(columns=["orig.ident",'tissuegroup', 'tumorgroup', 'site',"cells"])
crc01_obs['Glob_Sample'] = crc01_obs['Study'] + '-' + crc01_obs['Sample']
crc01_obs['Glob_Patient'] = crc01_obs['Study'] + '-' + crc01_obs['Patient']
crc01_obs = crc01_obs.set_index("Glob_CellID", drop=False)
adata_crc01.obs = crc01_obs.copy()
del crc01_obs
# reset analysis
adata_crc01.obsm = {}
adata_crc01.obsp = {}
adata_crc01.var = pd.DataFrame(index=adata_crc01.var.index)

In [24]:
### crc03 data
CurrentDir = DataRootDir / "crc03/GSE234804"
FileList = sorted(
    file for file in os.listdir(CurrentDir) if file.endswith(".h5ad")
)
adata_list = []

for file in FileList:
    name = Path(file).stem.split("_")[1]
    adata = sc.read_h5ad(CurrentDir / file)

    for layer_name in list(adata.layers.keys()):
        if layer_name == "scale.data":
            del adata.layers[layer_name]
            continue
        elif layer_name == "counts":
            adata.layers[layer_name] = ut.to_int32_counts(
                adata.layers[layer_name]
            )
            continue
        elif layer_name == "data":
            adata.layers[layer_name] = sp.sparse.csr_matrix(
                ut.to_float32(adata.layers[layer_name])
            )
            continue

    adata.X = adata.layers["data"]
    del adata.layers["data"]

    adata.obs["Study"] = "crc03"
    adata.obs["Sample"] = name
    adata.obs["Glob_CellID"] = "crc03-" + name + "_" + adata.obs.index
    adata.obs["Glob_Sample"] = "crc03-" + name

    prefix, num = re.findall(r"(\w+)(\d+)", name)[0]
    if prefix == "PC":
        tissue = "pm"
    elif prefix == "CRC":
        tissue = "crc"
    else:
        raise ValueError(f"Unknown tissue prefix: {prefix}")

    patient = "patient" + num
    adata.obs["Tissue"] = tissue
    adata.obs["Patient"] = patient
    adata.obs["Glob_Patient"] = "crc03-" + adata.obs["Patient"]
    adata.obs = adata.obs.set_index("Glob_CellID", drop=False)

    adata_list.append(adata)

adata_crc03 = ad.concat(
    adata_list,
    join="inner",
    merge="same",
    uns_merge="same"
)

In [25]:
print('Preview Data')
print('-----')
print(ut.preview_adata(adata_crc01))
print(ut.preview_adata(adata_crc03))
print('-----')
print('type of crc01: X {0} , raw counts {1}'.format(type(adata_crc01.X), type(adata_crc01.layers["counts"])))
print('type of crc03: X {0} , raw counts {1}'.format(type(adata_crc03.X), type(adata_crc03.layers["counts"])))
print('-----')
print('data type of crc01: X {0} , raw counts {1}'.format(adata_crc01.X.dtype, adata_crc01.layers["counts"].dtype))
print('data type of crc03: X {0} , raw counts {1}'.format(adata_crc03.X.dtype, adata_crc03.layers["counts"].dtype))

Preview Data
-----
shape: (47115, 13393)
X: csr_matrix, sparse=True
unique index: obs=True, var=True
obs: ['Sample', 'Tissue', 'Patient', 'Study', 'Glob_CellID', 'Glob_Sample', 'Glob_Patient']
var: []
layers: ['counts']
raw: False
obsm: []
obsp: []
uns: []
None
shape: (18101, 15795)
X: csr_matrix, sparse=True
unique index: obs=True, var=True
obs: ['Study', 'Sample', 'Glob_CellID', 'Glob_Sample', 'Tissue', 'Patient', 'Glob_Patient']
var: ['gene_ids']
layers: ['counts']
raw: False
obsm: []
obsp: []
uns: []
None
-----
type of crc01: X <class 'scipy.sparse._csr.csr_matrix'> , raw counts <class 'scipy.sparse._csr.csr_matrix'>
type of crc03: X <class 'scipy.sparse._csr.csr_matrix'> , raw counts <class 'numpy.ndarray'>
-----
data type of crc01: X float32 , raw counts int32
data type of crc03: X float32 , raw counts int32


#### Step two: Data Integration

In [38]:
# Merge crc01 and crc03 by shared gene symbols
adata_crc01.var['gene_name'] = adata_crc01.var_names
adata_crc03.var['gene_name'] = adata_crc03.var_names
adata_crc01.var['gene_id'] = ""
adata_crc03.var = adata_crc03.var.rename(columns={"gene_ids": "old_gene_id"})

adata_merged = ad.concat(
    [adata_crc03, adata_crc01],
    join="inner",
    merge="first" ,
    uns_merge=None,
)

ut.preview_adata(adata_merged)
print(f"X dtype: {adata_merged.X.dtype}")
print(f"counts dtype: {adata_merged.layers['counts'].dtype}")

shape: (65216, 10705)
X: csr_matrix, sparse=True
unique index: obs=True, var=True
obs: ['Study', 'Sample', 'Glob_CellID', 'Glob_Sample', 'Tissue', 'Patient', 'Glob_Patient']
var: ['old_gene_id', 'gene_name', 'gene_id']
layers: ['counts']
raw: False
obsm: []
obsp: []
uns: []
X dtype: float32
counts dtype: int32


In [39]:
adata_merged.var

,old_gene_id,gene_name,gene_id
TSPAN6,ENSG00000000003,TSPAN6,
DPM1,ENSG00000000419,DPM1,
FGR,ENSG00000000938,FGR,
CFH,ENSG00000000971,CFH,
FUCA2,ENSG00000001036,FUCA2,
...,...,...,...
TBCE,ENSG00000285053,TBCE,
AL109840.2,ENSG00000285091,AL109840.2,
AC016831.7,ENSG00000285106,AC016831.7,
ATXN7,ENSG00000285258,ATXN7,


#### Step three: unify vars
It is decided that gene symbol will be used as index, and ENSG id will be kept as back-ups
Format:
|gene_name|gene_id|
|-----|-----|
|gene A| ENSGXXXX|


Version: GRCh38.111


In [40]:
# Map merged gene symbols to GRCh38.111 ENSG IDs
mapping_path = Path.home() / "t_rna/database/GRCh38.111/gene_id_gene_name.csv"
gene_mapping = pd.read_csv(mapping_path, dtype=str)
gene_mapping["gene_id"] = gene_mapping["gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)

symbol_to_gene_ids = (
    gene_mapping.dropna(subset=["gene_name", "gene_id"])
    .groupby("gene_name")["gene_id"]
    .agg(lambda ids: ",".join(sorted(ids.unique())))
)

adata_merged.var["gene_id"] = adata_merged.var["gene_name"].map(
    symbol_to_gene_ids
)

mapped = adata_merged.var["gene_id"].notna()
multiple = adata_merged.var["gene_id"].str.contains(",", na=False)
adata_merged_unmapped_gene_names = adata_merged.var.index[~mapped].tolist()

print("adata_merged GRCh38.111 mapping")
print(f"mapped: {mapped.sum():,} / {adata_merged.n_vars:,} ({mapped.mean():.2%})")
print(f"cannot map to GRCh38.111: {(~mapped).sum():,}")
print(f"multiple ENSG IDs: {multiple.sum():,}")
print(
    "unmapped gene-name variable: "
    "adata_merged_unmapped_gene_names"
)

old_ids = adata_merged.var["old_gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)
new_ids = adata_merged.var["gene_id"]
both_ids_present = old_ids.notna() & new_ids.notna()
old_id_in_new_mapping = pd.Series(
    [
        pd.notna(old_id) and pd.notna(new_id) and old_id in new_id.split(",")
        for old_id, new_id in zip(old_ids, new_ids)
    ],
    index=adata_merged.var_names,
)

adata_merged_old_id_mismatch = adata_merged.var.loc[
    both_ids_present & ~old_id_in_new_mapping,
    ["gene_name", "old_gene_id", "gene_id"],
].copy()

print("\nOld-to-GRCh38.111 comparison")
print(f"both old and new IDs present: {both_ids_present.sum():,}")
print(f"old IDs present in new mapping: {old_id_in_new_mapping.sum():,}")
print(f"old/new ID mismatches: {len(adata_merged_old_id_mismatch):,}")
print("mismatch table: adata_merged_old_id_mismatch")

adata_merged GRCh38.111 mapping
mapped: 10,317 / 10,705 (96.38%)
cannot map to GRCh38.111: 388
multiple ENSG IDs: 18
unmapped gene-name variable: adata_merged_unmapped_gene_names

Old-to-GRCh38.111 comparison
both old and new IDs present: 10,317
old IDs present in new mapping: 10,309
old/new ID mismatches: 8
mismatch table: adata_merged_old_id_mismatch


### Data Export

In [44]:
export_dir = Path("../../result/curated/merged.h5ad")
adata_merged.write_h5ad(export_dir)